**Preparing Orders Data for Forecasting (PySpark)**

This notebook prepares last year’s orders data for downstream demand forecasting at Voltmart, an electronics e-commerce company.

Using PySpark, the raw orders_data.parquet file is cleaned and transformed into a modeling-ready table and saved as orders_data_clean.parquet.

In [ ]:
1. Setup & Data Load

from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession
    .builder
    .appName("voltmark_orders_data_cleaning")
    .getOrCreate()
)

orders_df = spark.read.parquet("orders_data.parquet")

In [ ]:
2. Time Filtering & time_of_day Feature

Cleaning rules applied here:

Remove orders placed between 00:00 and 05:00 (inclusive)

Convert order_date from timestamp → date

Create time_of_day:

"morning" → 5:00–12:00

"afternoon" → 12:00–18:00

"evening" → 18:00–24:00

hour_col = F.hour("order_date")


orders_clean = (
    orders_df
    .filter(hour_col >= 5)
    .withColumn(
        "time_of_day",
        F.when((hour_col >= 5) & (hour_col < 12), F.lit("morning"))
         .when((hour_col >= 12) & (hour_col < 18), F.lit("afternoon"))
         .otherwise(F.lit("evening"))
    )
    .withColumn("order_date", F.col("order_date").cast(T.DateType()))
)

In [ ]:
3. Product & Category Standardization

All product values → lowercase

All category values → lowercase

Remove rows where product contains "tv" (Voltmart no longer sells TVs)

orders_clean = (
    orders_clean
    .withColumn("product", F.lower("product"))
    .withColumn("category", F.lower("category"))
    .filter(~F.col("product").contains("tv"))
)

In [ ]:
4. Extract purchase_state from Address

purchase_state is derived from purchase_address, where the state abbreviation is always the second-to-last token in the address string.

orders_clean = (
    orders_clean
    .withColumn("address_tokens", F.split("purchase_address", " "))
    .withColumn(
        "purchase_state",
        F.col("address_tokens")[F.size("address_tokens") - 2]
    )
    .drop("address_tokens")
)

In [ ]:
5. Save Cleaned Dataset
(
    orders_clean
    .write
    .mode("overwrite")
    .parquet("orders_data_clean.parquet")
)

In [ ]:
6. Inspect Resulting Schema / Sample

(You can run these while exploring; they’re not required for the final pipeline.)

orders_clean.printSchema()
orders_clean.show(5, truncate=False)

In [ ]:
7. Stop Spark Session
spark.stop()